# P2 · N5a — Memorization Probe (clean run)

**Paper 2 — MARQ-Bench**

Method: Bordt, Nori & Caruana, *Elephants Never Forget: Testing Language Models
for Memorization of Tabular Data*, arXiv:2403.06644 (2024).

**What you are looking for is the contrast.** C1–C3 are public UCI corpora and
should show memorisation. C4 (NYC TLC, May 2026) post-dates every training
cutoff and should not.

---

### Fixed in this version

**A library bug invalidated the previous run.** `first_token_test` sets
`tabmem.config.max_tokens = num_digits` (i.e. 1) as a speed optimisation and
restores it afterwards — but without `try/finally`. When it raised
`ValueError: Failed to construct valid first tokens` on C4, the restore line was
never reached and **every subsequent call in the session was capped at one
token**. Responses came back empty and the harness reported a clean
"0/25 exact matches", which was the opposite of the truth: a direct call
reproduced row 3 of Bank Marketing character-for-character.

This notebook resets `max_tokens` before every test, so a failure can no longer
contaminate what follows.

**Ordering fixed.** The one-time maintenance cell that clears old results now
runs *before* the sweep. In the previous version it sat after it, so Run All
would have deleted the results it had just paid for.

**Diagnostics removed.** They did their job.

---

**Expected:** C4's `first_token` will fail again with
`ValueError: Failed to construct valid first tokens`. That is correct — C4's
leading column is `VendorID`, which is near-binary, so no usable token exists.
Report it as *test not applicable to C4*.

**Runtime:** 30–60 min. **Cost:** ~$5–8 for the full matrix.

## 1 · Install, then RESTART

Run this, then **Runtime → Restart session**, then run from the top. The
install is skipped on the second pass.

In [ ]:
try:
    import tabmemcheck
    print('tabmemcheck', tabmemcheck.__version__, '- installed, continue')
except ImportError:
    %pip install -q tabmemcheck anthropic
    print('INSTALLED. Now: Runtime > Restart session, then run from the top.')

## 2 · Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3 · Setup

In [ ]:
import sys, os, json, datetime
from pathlib import Path
import pandas as pd

ROOT      = Path('/content/drive/MyDrive/Paper2_RuleAuthorship')
MODULES   = ROOT / 'Notebooks'
ARTIFACTS = ROOT / 'artifacts'
PROBE_DIR = ROOT / 'memorization'

assert ROOT.exists() and MODULES.exists()
for d in (ARTIFACTS, PROBE_DIR):
    d.mkdir(parents=True, exist_ok=True)
if str(MODULES) not in sys.path:
    sys.path.insert(0, str(MODULES))

import llmauth_census as C
import tabmemcheck as tabmem
from tabmemcheck.llm import LLM_Interface

PATTERNS = {
    'bank_marketing':   ['bankfull', 'bank-full', 'bank_full'],
    'diabetes_130us':   ['diabetic_data', 'diabetes'],
    'online_retail_ii': ['online_retail', 'online retail', 'retail'],
    'nyc_tlc_yellow':   ['yellow_tripdata'],
}

def discover_data():
    files = [p for p in ROOT.rglob('*')
             if p.is_file() and p.suffix.lower() in ('.csv','.parquet','.xlsx')]
    out = {}
    for corpus, frags in PATTERNS.items():
        hits = [p for p in files if any(f in p.name.lower() for f in frags)]
        hits.sort(key=lambda p: p.stat().st_size, reverse=True)
        if hits: out[corpus] = hits[0]
    return out

DATA_PATHS = discover_data()
print('tabmemcheck', tabmem.__version__, '| census', C.CENSUS_VERSION)
print('corpora found:', list(DATA_PATHS))

## 4 · API key (Colab Secrets, key icon in the sidebar)

In [ ]:
from google.colab import userdata
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
print('key loaded:', bool(os.environ.get('ANTHROPIC_API_KEY')))

## 5 · Anthropic adapter

`tabmemcheck` ships adapters for OpenAI and Gemini only. `LLM_Interface` is a
two-method abstraction, so an Anthropic adapter is short. The system message is
routed to Anthropic's top-level parameter, where it belongs.

In [ ]:
class AnthropicLLM(LLM_Interface):
    chat_mode = True

    def __init__(self, model_id, api_key=None):
        import anthropic
        self.model_id = model_id
        self._anthropic = anthropic
        self._client = anthropic.Anthropic(
            api_key=api_key or os.environ.get('ANTHROPIC_API_KEY'))
        self._temperature_ok = None
        self.calls = 0
        self.in_tok = 0
        self.out_tok = 0

    def __repr__(self):
        return self.model_id

    def reset_counters(self):
        self.calls = self.in_tok = self.out_tok = 0

    def chat_completion(self, messages, temperature: float, max_tokens: int):
        system_parts = [m['content'] for m in messages if m['role'] == 'system']
        convo = [{'role': m['role'], 'content': m['content']}
                 for m in messages if m['role'] != 'system']
        kw = dict(model=self.model_id, max_tokens=max_tokens, messages=convo)
        if system_parts:
            kw['system'] = '\n'.join(system_parts)
        if self._temperature_ok is not False:
            kw['temperature'] = temperature
        try:
            resp = self._client.messages.create(**kw)
        except self._anthropic.BadRequestError as exc:
            if 'temperature' not in str(exc).lower() or self._temperature_ok is False:
                raise
            self._temperature_ok = False
            kw.pop('temperature', None)
            resp = self._client.messages.create(**kw)
        else:
            if self._temperature_ok is None:
                self._temperature_ok = True
        self.calls += 1
        self.in_tok  += getattr(resp.usage, 'input_tokens', 0) or 0
        self.out_tok += getattr(resp.usage, 'output_tokens', 0) or 0
        return ''.join(b.text for b in resp.content
                       if getattr(b, 'type', '') == 'text')

print('adapter defined')

## 6 · Write the probe CSVs

Original file order, original tokens, via `load_corpus` so nothing is
re-encoded. A probe against re-encoded data would test memorisation of
something the model never saw.

In [ ]:
PROBE_ROWS = 2000

probe_files = {}
for cid, path in DATA_PATHS.items():
    df, _ = C.load_corpus(cid, path)
    out = PROBE_DIR / f'{cid}_probe.csv'
    df.head(PROBE_ROWS).to_csv(out, index=False)
    probe_files[cid] = out
    print(f'{cid:<18} {len(df):>9,} rows -> first {PROBE_ROWS:,} written')

## 7 · One-time maintenance — clear the poisoned results

The previous run was corrupted by the `max_tokens` leak, so those JSONs must go
before the sweep. **This runs before the main loop deliberately** — placed after
it, a Run All would delete results it had just paid for.

Set `CLEAR_OLD = False` after the first clean run, so a later Run All cannot
wipe good results.

In [ ]:
CLEAR_OLD = True          # set to False once you have one clean run

tabmem.config.max_tokens = 1000
print('config.max_tokens =', tabmem.config.max_tokens)

if CLEAR_OLD:
    n = 0
    for f in PROBE_DIR.glob('*__*.json'):
        f.unlink(); n += 1
    print(f'cleared {n} previous result file(s)')
else:
    print('CLEAR_OLD is False — existing results kept and will be reused')

## 8 · Scope

Full matrix: 4 corpora × 2 models ≈ 440 calls, roughly $5–8.

In [ ]:
MODELS  = ['claude-sonnet-4-5-20250929', 'claude-haiku-4-5-20251001']
CORPORA = list(probe_files)

print(f'planned: {len(MODELS)} models x {len(CORPORA)} corpora '
      f'~= {len(MODELS)*len(CORPORA)*55} calls')
print('  models :', MODELS)
print('  corpora:', CORPORA)

## 9 · Run

Resumable and cached **per test**, so an interruption costs only the test that
failed. `max_tokens` is reset before every test, so a failure cannot
contaminate what follows.

In [ ]:
results = {}
for model_id in MODELS:
    llm = AnthropicLLM(model_id)
    for cid in CORPORA:
        key = f'{model_id}|{cid}'
        out_json = PROBE_DIR / f'{cid}__{model_id}.json'
        r = json.loads(out_json.read_text()) if out_json.exists() else {}

        print(f'\n{"="*66}\n{model_id}  x  {cid}\n{"="*66}')
        path = str(probe_files[cid])
        llm.reset_counters()          # per corpus x model accounting

        for name, fn in [
            ('feature_names',  lambda: tabmem.feature_names_test(path, llm)),
            ('header',         lambda: tabmem.header_test(path, llm)),
            ('row_completion', lambda: tabmem.row_completion_test(path, llm, num_queries=25)),
            ('first_token',    lambda: tabmem.first_token_test(path, llm, num_queries=25)),
        ]:
            if name in r:
                print(f'  [cached] {name}')
                continue

            # CRITICAL: first_token_test leaks max_tokens=1 when it raises.
            tabmem.config.max_tokens = 1000

            try:
                r[name] = fn()
                r.pop(name + '_error', None)
            except Exception as e:
                msg = f'{type(e).__name__}: {str(e)[:300]}'
                r[name + '_error'] = msg
                print(f'  !! {name}: {msg}')
                if 'credit balance' in msg.lower() or 'rate_limit' in msg.lower():
                    print('\n  STOPPING — account issue, not a code issue.')
                    out_json.write_text(json.dumps(r, indent=2, default=str))
                    raise SystemExit
            finally:
                tabmem.config.max_tokens = 1000     # restore no matter what

            r['_model'], r['_corpus'] = model_id, cid
            r['_calls'], r['_input_tokens'], r['_output_tokens'] = (
                llm.calls, llm.in_tok, llm.out_tok)
            r['_run_at_utc'] = datetime.datetime.now(
                datetime.timezone.utc).isoformat(timespec='seconds')
            out_json.write_text(json.dumps(r, indent=2, default=str))

        results[key] = r
        print(f'  {llm.calls} calls, {llm.in_tok:,} in / {llm.out_tok:,} out tokens')

## 10 · Sanity check — did the responses have content?

The previous run's failure mode was empty responses with one output token per
call. This makes that impossible to miss.

In [ ]:
print(f'{"corpus":<18}{"model":<30}{"calls":>7}{"out tok":>9}{"tok/call":>10}')
print('-' * 74)
for key, r in sorted(results.items()):
    model, cid = key.split('|')
    calls, out = r.get('_calls', 0), r.get('_output_tokens', 0)
    per = out / calls if calls else 0
    flag = '  <-- SUSPICIOUS' if 0 < per < 5 else ''
    print(f'{cid:<18}{model:<30}{calls:>7}{out:>9,}{per:>10.1f}{flag}')

print('\nHealthy runs average tens of tokens per call. Anything near 1 means')
print('the max_tokens leak recurred and those results must be discarded.')

## 11 · Summary

C1–C3 are public: memorisation expected. C4 post-dates every training cutoff:
memorisation should be absent. That contrast is what §4.3 rests on.

In [ ]:
print(f'{"corpus":<18}{"model":<30}{"control?":<10}results')
print('-' * 95)
for key, r in sorted(results.items()):
    model, cid = key.split('|')
    ctrl = 'CONTROL' if cid == 'nyc_tlc_yellow' else 'public'
    vals = ', '.join(f'{k}={v}' for k, v in r.items()
                     if not k.startswith('_') and not k.endswith('_error'))
    errs = [k for k in r if k.endswith('_error')]
    print(f'{cid:<18}{model:<30}{ctrl:<10}{vals[:60]}'
          + (f'  [failed: {", ".join(e[:-6] for e in errs)}]' if errs else ''))

(ARTIFACTS / 'N5a_memorization.json').write_text(
    json.dumps(results, indent=2, default=str))
print(f'\nwrote {ARTIFACTS / "N5a_memorization.json"}')
print('\nSend that file. Check section 10 first — if tokens/call is near 1,')
print('do not trust the numbers.')